### `Step-3` : Image Text Filter
- After applying a resolution filter, I noticed that some images contained text, which could negatively affect the model's learning accuracy. Therefore, I applied a text filter to remove images containing text. After this step, I downloaded the image data folder from HDFS to the local disk. The following code lines were executed in the Ubuntu terminal to perform the download:
  
  - rm -rf /home/student/Desktop/dangerous_animals
  
  - hdfs dfs -get /user/student/dangerous_animals /home/student/Desktop/datasets/
  
- When I manually reviewed the images, I realized that some did not represent animals or were duplicates. I manually removed these types of images to improve dataset quality.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.image import ImageSchema
import numpy as np
import os

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Image Text Filter") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# HDFS folder path
hdfs_path = "hdfs://localhost:9000/user/student/dangerous_animals/datasets/"

# Function to detect potential text in an image
def has_text(image_data):
    try:
        image_array = np.array(image_data)
        gray = np.mean(image_array, axis=-1)  # Convert to grayscale
        
        # Edge detection using gradient change
        gradient = np.abs(np.diff(gray, axis=1))
        edge_intensity = np.mean(gradient)
        
        # Threshold for text detection
        return edge_intensity > 15
    except Exception as e:
        print(f"Error processing image: {e}")
        return False

# Load images from HDFS
image_df = spark.read.format("image").load(hdfs_path)

# Delete images with text-like patterns
def delete_images_with_text():
    deleted_count = 0
    
    for row in image_df.collect():
        image_path = row['image']['origin']
        if has_text(row['image']['data']):
            os.system(f"hdfs dfs -rm {image_path}")
            print(f"Deleted: {image_path}")
            deleted_count += 1

    print(f"\nTotal images deleted: {deleted_count}")

# Start deletion
delete_images_with_text()
spark.stop()
